# Do Surgeons Run on Time While the System Fails Them?

**Hypothesis:** The bulk of OR time overruns accumulate *before* the procedure starts (prep & patient positioning), not during the procedure itself.

**Method:** We compute each step's gap (actual − planned) **independently** using planned durations, so a late prep start doesn't artificially inflate the in-OR gap.

- **Prep gap** = (actual patient in OR − actual prep start) − (planned patient in OR − planned prep start)
- **In-OR gap** = (actual patient leaves OR − actual patient in OR) − (planned patient leaves OR − planned patient in OR)

Each step is measured against its own planned *duration*, not against an absolute clock. This eliminates cascading effects.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
AMBER = '#E9C46A'
GREY = '#8D99AE'
RED = '#E63946'
GREEN = '#2A9D8F'

In [ ]:
DATA_PATH = 'completed_operations.csv'  # <-- update this

df = pd.read_csv(DATA_PATH, sep=';', encoding='utf-8')

for col in ['Planlagt stue klargøring start', 'Stue klargøring start',
            'Patient på stuen (Planlagt)', 'Patient på stuen',
            'Patient forlader stuen (Planlagt)', 'Patient forlader stuen']:
    df[col] = pd.to_datetime(df[col], format='mixed', errors='coerce')

df['Procedure_Name'] = df['Procedure - Tekst & ID'].str.replace(r'\s*\[.*\]', '', regex=True).str.strip()

print(f'{len(df):,} rows loaded')

In [ ]:
# --- Compute INDEPENDENT step durations (no cascading) ---
# Each gap compares actual DURATION of a step vs planned DURATION of the same step.

def dur(df, a, b):
    return (df[b] - df[a]).dt.total_seconds() / 60

# Planned durations
df['planned_prep_dur'] = dur(df, 'Planlagt stue klargøring start', 'Patient på stuen (Planlagt)')
df['planned_inOR_dur'] = dur(df, 'Patient på stuen (Planlagt)', 'Patient forlader stuen (Planlagt)')

# Actual durations
df['actual_prep_dur'] = dur(df, 'Stue klargøring start', 'Patient på stuen')
df['actual_inOR_dur'] = dur(df, 'Patient på stuen', 'Patient forlader stuen')

# Gaps: how much longer each step took vs its own plan
df['prep_gap'] = df['actual_prep_dur'] - df['planned_prep_dur']
df['inOR_gap'] = df['actual_inOR_dur'] - df['planned_inOR_dur']

# Drop rows where either gap is missing or clearly erroneous
valid = df.dropna(subset=['prep_gap', 'inOR_gap']).copy()
valid = valid[(valid['prep_gap'].abs() < 300) & (valid['inOR_gap'].abs() < 300)]

print(f'{len(valid):,} rows with valid independent step gaps')

In [ ]:
# --- Global summary across ALL procedures ---
print('GLOBAL MEAN GAP (actual duration − planned duration):')
print(f'  Prep step:  {valid["prep_gap"].mean():+.1f} min  (median {valid["prep_gap"].median():+.1f})')
print(f'  In-OR step: {valid["inOR_gap"].mean():+.1f} min  (median {valid["inOR_gap"].median():+.1f})')
print(f'\n  → Prep accounts for {valid["prep_gap"].mean() / (valid["prep_gap"].mean() + valid["inOR_gap"].mean()) * 100:.0f}% of total overrun')

In [ ]:
# --- Per-procedure breakdown (procedures with >= 30 valid cases) ---
MIN_N = 30

proc_stats = (
    valid
    .groupby('Procedure_Name')
    .agg(
        n=('prep_gap', 'count'),
        prep_mean=('prep_gap', 'mean'),
        inOR_mean=('inOR_gap', 'mean'),
    )
    .query(f'n >= {MIN_N}')
)

proc_stats['total_mean'] = proc_stats['prep_mean'] + proc_stats['inOR_mean']
proc_stats['prep_share'] = proc_stats['prep_mean'] / proc_stats['total_mean'] * 100
proc_stats = proc_stats.sort_values('prep_mean', ascending=False)

print(f'{len(proc_stats)} procedures with >= {MIN_N} cases')
print(f'{proc_stats["prep_mean"].gt(proc_stats["inOR_mean"]).sum()} have larger prep gap than in-OR gap')
proc_stats.round(1).head(15)

In [ ]:
# --- Main chart: prep gap vs in-OR gap per procedure ---
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(proc_stats['inOR_mean'], proc_stats['prep_mean'],
           s=proc_stats['n'] * 0.3, alpha=0.6, color=RED, edgecolors='white', linewidth=0.5)

# Diagonal = equal contribution
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, '--', color='#aaa', linewidth=1, zorder=0, label='Equal gap')

ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax.axvline(0, color='black', linewidth=0.5, linestyle=':')

ax.set_xlabel('In-OR Step Gap (actual − planned duration, min)')
ax.set_ylabel('Prep Step Gap (actual − planned duration, min)')
ax.set_title('Prep Gap vs In-OR Gap by Procedure\n'
             'Points above the diagonal = prep is the bigger problem',
             fontweight='bold')
ax.legend(fontsize=9)

# Label the most extreme points
for _, row in proc_stats.nlargest(5, 'prep_mean').iterrows():
    ax.annotate(row.name[:25], (row['inOR_mean'], row['prep_mean']),
                fontsize=7, alpha=0.8, xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig('prep_vs_inor_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Histogram: what share of overrun comes from prep? ---
# Only for procedures with positive total overrun
overrun = proc_stats[proc_stats['total_mean'] > 0].copy()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(overrun['prep_share'].clip(0, 100), bins=20, color=AMBER, edgecolor='white')
ax.axvline(50, color='black', linewidth=1, linestyle='--', label='50% line')
ax.axvline(overrun['prep_share'].median(), color=RED, linewidth=2,
           label=f'Median: {overrun["prep_share"].median():.0f}%')
ax.set_xlabel('% of total overrun from prep step')
ax.set_ylabel('Number of procedures')
ax.set_title('How Often Is Prep the Dominant Source of Overrun?', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('prep_share_histogram.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Of {len(overrun)} procedures with net overrun:')
print(f'  {(overrun["prep_share"] > 50).sum()} ({(overrun["prep_share"] > 50).mean()*100:.0f}%) have prep as the majority source')

In [ ]:
# --- Final verdict ---
prep_dominant = (proc_stats['prep_mean'] > proc_stats['inOR_mean']).mean() * 100

print('='*60)
print('VERDICT')
print('='*60)
print(f'Across {len(proc_stats)} procedures (≥{MIN_N} cases each):')
print(f'  {prep_dominant:.0f}% have a larger prep gap than in-OR gap.')
print(f'  Global mean prep gap:  {valid["prep_gap"].mean():+.1f} min')
print(f'  Global mean in-OR gap: {valid["inOR_gap"].mean():+.1f} min')
print()
if prep_dominant > 60:
    print('  ✓ The pattern holds broadly: surgeons run close to plan,')
    print('    the system loses time getting them started.')
elif prep_dominant > 40:
    print('  ~ Mixed: prep overruns are common but not universal.')
else:
    print('  ✗ The pattern does NOT hold generally — in-OR overruns dominate.')